# Module 1: Probability & Statistics for Quantitative Finance


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from scipy.stats import norm, binom, poisson, uniform, lognorm

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

## 1. Random Variables

A **random variable** $X$ is a function that maps outcomes of a random experiment to numerical values. Formally, it is a measurable function from a sample space $\Omega$ to the real numbers $\mathbb{R}$.

### Types of Random Variables

| Type | Description | Example |
|---|---|---|
| **Discrete** | Takes countable values | Number of defaults in a portfolio |
| **Continuous** | Takes values in an interval | Daily stock return |
| **Mixed** | Combination of both | A bond that defaults (0) or pays (continuous return) |

### Why Random Variables in Finance?

In finance, **uncertainty is the central object of study**. Whether we are modeling:
- The price of a stock tomorrow
- Whether a counterparty defaults
- The volatility of an option's underlying

...we are always dealing with quantities whose outcomes are uncertain. Random variables give us the mathematical language to reason about them precisely.

# 2 Probability Distributions


## 2. Probability Distributions

### The Probability Density Function (PDF)

For a **continuous** random variable, the probability of $X$ lying in a small interval $[x, x+dx]$ is $p(x)\,dx$. More generally:

$$\text{Prob}(a < X < b) = \int_{a}^{b} p(x)\, dx$$

The function $p(x)$ is called the **Probability Density Function (PDF)** and must satisfy:
1. $p(x) \geq 0$ — probabilities are non-negative
2. $\int_{-\infty}^{\infty} p(x)\, dx = 1$ — total probability is 1

### The Cumulative Distribution Function (CDF)

The **CDF** $F(x)$ answers the question: *what is the probability that $X$ takes a value no larger than $x$?*

$$F(x) = \text{Prob}(X \leq x) = \int_{-\infty}^{x} p(x')\, dx'$$

Key properties:
- $F(-\infty) = 0$, $F(+\infty) = 1$
- $F$ is non-decreasing
- The PDF is the derivative: $p(x) = \frac{dF(x)}{dx}$
- Useful for computing interval probabilities: $\text{Prob}(a < X < b) = F(b) - F(a)$

> **Why is the CDF often preferred empirically?** Because it can be estimated directly from data (as the empirical CDF) without any smoothing, unlike the PDF which requires kernel density estimation or binning.

### Change of Variables

If $Y = y(X)$ is a monotone transformation of $X$, then the density of $Y$ is:

$$g(y) = \frac{p(x)}{|dy/dx|}$$

This is the **Jacobian rule** — it ensures that probability is preserved: $p(x)\,dx = g(y)\,dy$. The absolute value is needed to keep $g(y) \geq 0$ when $y$ is decreasing in $x$.

**Example:** If $X \sim \mathcal{N}(\mu, \sigma^2)$ and $Y = e^X$, then $Y$ is **lognormally distributed** — a key relationship in finance since log-returns are (approximately) Gaussian while prices are lognormal.

In [ ]:
"""
Illustrate PDF vs CDF for a standard Gaussian and the change-of-variables rule.
"""
x = np.linspace(-4, 4, 500)
pdf = norm.pdf(x)
cdf = norm.cdf(x)

a, b = -1.0, 1.5

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# PDF with shaded area
axes[0].plot(x, pdf, 'steelblue', linewidth=2, label='$p(x)$')
mask = (x >= a) & (x <= b)
axes[0].fill_between(x[mask], pdf[mask], alpha=0.35, color='steelblue',
                     label=f'$P({a}<X<{b})={norm.cdf(b)-norm.cdf(a):.3f}$')
axes[0].axvline(a, color='gray', linestyle='--', linewidth=1)
axes[0].axvline(b, color='gray', linestyle='--', linewidth=1)
axes[0].set(title='Probability Density Function (PDF)', xlabel='$x$', ylabel='$p(x)$')
axes[0].legend()

# CDF with same interval highlighted
axes[1].plot(x, cdf, 'steelblue', linewidth=2, label='$F(x)$')
axes[1].plot([a, b], [norm.cdf(a), norm.cdf(a)], 'r--', linewidth=1.5)
axes[1].plot([a, b], [norm.cdf(b), norm.cdf(b)], 'r--', linewidth=1.5)
axes[1].annotate('', xy=(b, norm.cdf(b)), xytext=(b, norm.cdf(a)),
                 arrowprops=dict(arrowstyle='<->', color='red', lw=2))
axes[1].text(b + 0.1, (norm.cdf(a)+norm.cdf(b))/2,
             f'$F({b})-F({a})$\n$={norm.cdf(b)-norm.cdf(a):.3f}$', color='red', va='center')
axes[1].set(title='Cumulative Distribution Function (CDF)', xlabel='$x$', ylabel='$F(x)$')
axes[1].legend()

plt.suptitle('PDF and CDF of the Standard Normal $\\mathcal{N}(0,1)$', fontsize=12)
plt.tight_layout()
plt.show()

# 3 Expectation and moments

## 3. Expectation and Moments

### Expected Value

The **expected value** $\mathbb{E}[f(X)]$ is the probability-weighted average of $f$ over all outcomes:

$$\mathbb{E}[f(X)] = \begin{cases} \sum_k f(x_k)\, p(x_k) & \text{(discrete)} \\ \int_{-\infty}^{\infty} f(x)\, p(x)\, dx & \text{(continuous)} \end{cases}$$

The expectation operator is **linear**: $\mathbb{E}[af(X) + bg(X)] = a\,\mathbb{E}[f] + b\,\mathbb{E}[g]$.

### Moments

The **$\ell$-th moment** is $\mu_\ell = \mathbb{E}[X^\ell]$. The first few moments encode the shape of the distribution completely:

$$\mu_1 = \mathbb{E}[X] \equiv \mu \quad \text{(mean)}$$
$$\mu_2 = \mathbb{E}[X^2] \quad \Rightarrow \quad \sigma^2 = \mathbb{E}[(X-\mu)^2] = \mu_2 - \mu_1^2 \quad \text{(variance)}$$

### Higher Moments: Shape Descriptors

| Moment | Formula | Interpretation |
|---|---|---|
| **Variance** | $\sigma^2 = \mathbb{E}[(X-\mu)^2]$ | Average squared deviation from mean |
| **Skewness** | $s = \mathbb{E}\!\left[\left(\frac{X-\mu}{\sigma}\right)^3\right]$ | Asymmetry: $s>0$ = right tail, $s<0$ = left tail |
| **Excess Kurtosis** | $k = \mathbb{E}\!\left[\left(\frac{X-\mu}{\sigma}\right)^4\right] - 3$ | Tail weight relative to Gaussian ($k=0$ for Gaussian) |

### Why Higher Moments Matter in Finance

- A portfolio manager with $s < 0$ (negative skew) faces **crash risk**: small frequent gains, but occasional large losses.
- Positive excess kurtosis ($k > 0$, **fat tails**) means extreme events are far more likely than a Gaussian would predict — this is empirically observed in financial returns and is why Black-Scholes underprices out-of-the-money options.
- The Gaussian has $s = 0$ and $k = 0$: it is the **benchmark** against which all other distributions are measured.

In [ ]:
"""
Visualise skewness and kurtosis using parametric distributions.
"""
x = np.linspace(-5, 5, 600)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Skewness: compare normal vs skewed distributions
axes[0].plot(x, norm.pdf(x), 'steelblue', lw=2, label='Gaussian ($s=0$)')
# Positive skew: chi-squared(df=4) shifted and scaled
chi2_rv = stats.chi2(df=4)
x_c = np.linspace(0, 15, 600)
pdf_c = chi2_rv.pdf(x_c)
mu_c, sig_c = chi2_rv.mean(), chi2_rv.std()
axes[0].plot((x_c - mu_c) / sig_c, pdf_c * sig_c, 'tomato', lw=2,
             label=f'Right-skewed ($s>0$, $\\chi^2_4$)')
# Negative skew: mirror
axes[0].plot(-(x_c - mu_c) / sig_c, pdf_c * sig_c, 'seagreen', lw=2,
             label='Left-skewed ($s<0$)')
axes[0].set(title='Effect of Skewness', xlabel='Standardized $x$', ylabel='Density',
            xlim=(-4, 4))
axes[0].legend()
axes[0].axvline(0, color='gray', linestyle=':', linewidth=1)

# Kurtosis: Gaussian vs fat-tailed (t-distribution)
axes[1].plot(x, norm.pdf(x), 'steelblue', lw=2, label='Gaussian ($k=0$)')
for df, color in [(30, 'goldenrod'), (5, 'tomato'), (3, 'purple')]:
    t_rv = stats.t(df=df)
    excess_k = 6/(df-4) if df > 4 else float('inf')
    label = f'$t$({df}) — {"$k≈$" + f"{excess_k:.1f}" if df > 4 else "fat tail"}'
    axes[1].plot(x, t_rv.pdf(x), color=color, lw=2, label=label)
axes[1].set(title='Effect of Excess Kurtosis (Fat Tails)',
            xlabel='Standardized $x$', ylabel='Density', ylim=(0, 0.45))
axes[1].legend()

plt.suptitle('Shape of Distributions: Skewness and Kurtosis', fontsize=12)
plt.tight_layout()
plt.show()

# Numerical verification
print(f"{'Distribution':<20} {'Mean':>8} {'Std':>8} {'Skew':>8} {'Kurt':>8}")
print('-' * 56)
for label, dist in [('Gaussian N(0,1)', norm()),
                    ('t(df=5)',  stats.t(df=5)),
                    ('t(df=3)',  stats.t(df=3)),
                    ('chi2(4) standardized', None)]:
    if label.startswith('chi2'):
        sample = (stats.chi2(4).rvs(100_000) - 4) / np.sqrt(8)
        m, s, sk, ku = np.mean(sample), np.std(sample), stats.skew(sample), stats.kurtosis(sample)
    else:
        sample = dist.rvs(100_000)
        m, s, sk, ku = np.mean(sample), np.std(sample), stats.skew(sample), stats.kurtosis(sample)
    print(f"{label:<20} {m:>8.3f} {s:>8.3f} {sk:>8.3f} {ku:>8.3f}")

# 4 Covariance and correlation

## 4. Covariance and Correlation

### Covariance

For two random variables $X$ and $Y$, the **covariance** measures how much they move together:

$$\text{Cov}(X, Y) = \mathbb{E}[(X - \mu_X)(Y - \mu_Y)] = \mathbb{E}[XY] - \mu_X \mu_Y$$

- $\text{Cov}(X,Y) > 0$: $X$ and $Y$ tend to move in the **same** direction
- $\text{Cov}(X,Y) < 0$: they tend to move in **opposite** directions
- $\text{Cov}(X,Y) = 0$: **no linear** relationship (but could still be dependent!)

### Correlation

The **correlation** normalizes the covariance to lie in $[-1, +1]$:

$$\rho(X,Y) = \frac{\text{Cov}(X,Y)}{\sqrt{\text{Var}(X)\,\text{Var}(Y)}} \in [-1, +1]$$

- $\rho = +1$: perfect positive linear relationship
- $\rho = -1$: perfect negative linear relationship
- $\rho = 0$: no **linear** dependence

### Critical Warning: Zero Correlation ≠ Independence

Zero covariance only implies no *linear* relationship. Consider $X \sim \mathcal{N}(0,1)$ and $Y = X^2$. Then $\text{Cov}(X,Y) = \mathbb{E}[X^3] - \mathbb{E}[X]\mathbb{E}[X^2] = 0$ (odd moments of a symmetric distribution vanish), yet $Y$ is completely determined by $X$.

### Covariance Matrix

For $N$ assets, pairwise covariances form the **covariance matrix** $\Sigma$:

$$\Sigma_{ij} = \text{Cov}(R_i, R_j), \quad \Sigma_{ii} = \text{Var}(R_i) = \sigma_i^2$$

The covariance matrix is **symmetric** and **positive semi-definite** — this is the mathematical constraint that portfolio variance $\sigma_p^2 = \mathbf{w}^T \Sigma\, \mathbf{w} \geq 0$ for any weight vector $\mathbf{w}$.

In [ ]:
"""
Visualise correlation and the difference between zero-correlation and independence.
"""
N_samp = 2000
X = np.random.randn(N_samp)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# Six scatter plots with different correlation levels
configs = [
    ('$\\rho = +0.9$', 0.9),
    ('$\\rho = +0.3$', 0.3),
    ('$\\rho = 0$ (independent)', 0.0),
    ('$\\rho = -0.3$', -0.3),
    ('$\\rho = -0.9$', -0.9),
    ('$\\rho = 0$ but NOT independent\n$Y = X^2$', None),
]

for ax, (title, rho) in zip(axes.flatten(), configs):
    x_s = np.random.randn(N_samp)
    if rho is None:
        y_s = x_s**2
    else:
        y_s = rho * x_s + np.sqrt(1 - rho**2) * np.random.randn(N_samp)
    emp_rho = np.corrcoef(x_s, y_s)[0, 1]
    ax.scatter(x_s, y_s, alpha=0.15, s=6, color='steelblue')
    ax.set(title=title, xlabel='$X$', ylabel='$Y$')
    ax.text(0.05, 0.93, f'Empirical $\\rho={emp_rho:.2f}$',
            transform=ax.transAxes, fontsize=9,
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.suptitle('Scatter Plots for Different Correlation Levels', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Common Distributions

The following distributions appear throughout quantitative finance. Each is characterized by its PDF, moments, and a typical application.

## Uniform distribution

### 5.1 Uniform Distribution

The **uniform distribution** on $[a, b]$ assigns equal probability density to every point in the interval. In its standard form on $[0,1]$:

$$p(x) = \begin{cases} 1, & x \in [0,1] \\ 0, & \text{otherwise} \end{cases}, \qquad F(x) = x$$

**Moments (standard form):**

$$\mu = \frac{1}{2}, \qquad \mu_\ell = \frac{1}{\ell+1}, \qquad \sigma^2 = \frac{1}{12}$$

**Use in finance:** Uniform random variables are the raw material for generating samples from *any* distribution via the **inverse CDF method**: if $U \sim \text{Uniform}(0,1)$ then $F^{-1}(U)$ has CDF $F$. This is the backbone of Monte Carlo simulation.

## Binomial distribution

### 5.2 Binomial Distribution

The **binomial distribution** $B(n, p)$ counts the number of "successes" in $n$ independent Bernoulli trials, each with success probability $p$:

$$f(k; n, p) = \binom{n}{k} p^k (1-p)^{n-k}, \qquad k = 0, 1, \dots, n$$

**Moments:**

$$\mu = np, \qquad \sigma^2 = np(1-p) = npq$$

**Intuition for the mean:** Each trial contributes $p$ to the expected number of successes (by linearity of expectation), so the total is $np$.

**Use in finance:** Binomial trees (Cox-Ross-Rubinstein model) discretize the price process into up/down moves — the number of up-moves in $n$ steps is binomially distributed. Also used for credit modelling (number of defaults in a portfolio of $n$ bonds).

## Gaussian distribution

### 5.3 Gaussian (Normal) Distribution

The **Gaussian** is the most important distribution in probability and finance:

$$p(x) = \frac{1}{\sqrt{2\pi\sigma^2}} e^{-(x-\mu)^2 / 2\sigma^2}$$

Standardizing $z = (x-\mu)/\sigma$ reduces it to $\mathcal{N}(0,1)$ with $p(z) = \frac{1}{\sqrt{2\pi}}e^{-z^2/2}$.

**Key integrals:**

$$\int_{-\infty}^{\infty} e^{-ax^2} dx = \sqrt{\frac{\pi}{a}}, \qquad \int_{-\infty}^{\infty} x^2 e^{-ax^2} dx = \frac{1}{2}\sqrt{\frac{\pi}{a^3}}$$

The second integral is obtained by differentiating the first with respect to $a$ — a trick worth remembering.

**CDF:**

$$F(x) = \Phi\!\left(\frac{x-\mu}{\sigma}\right), \qquad \Phi(z) = \frac{1}{\sqrt{2\pi}}\int_{-\infty}^z e^{-z'^2/2} dz'$$

$\Phi$ has no closed form but is tabulated. Key values: $\Phi(1.65) \approx 0.95$, $\Phi(1.96) \approx 0.975$, $\Phi(2.33) \approx 0.99$.

**Skewness = 0, Excess kurtosis = 0.** The Gaussian is the benchmark of "normality" — any deviation from these values signals non-Gaussian behavior.

**Use in finance:** Log-returns are approximately Gaussian over short horizons. The CLT justifies its ubiquity for sums of many small shocks.

## Lognormal distribution

### 5.4 Lognormal Distribution

If $X \sim \mathcal{N}(\mu, \sigma^2)$ and $Y = e^X$, then $Y$ is **lognormally distributed**. Using the change-of-variables formula with $x = \log y$, $|dy/dx| = y$:

$$g(y) = \frac{p(\log y)}{y} = \frac{1}{y\sqrt{2\pi\sigma^2}} \exp\!\left(-\frac{(\log y - \mu)^2}{2\sigma^2}\right), \qquad y > 0$$

**Moments:**

$$\mathbb{E}[Y] = e^{\mu + \sigma^2/2}, \qquad \text{Var}(Y) = \left(e^{\sigma^2} - 1\right) e^{2\mu + \sigma^2}$$

**Key insight:** $\mathbb{E}[Y] > e^\mu$ — the mean of the lognormal is **above** $e^\mu$ (the median), by a factor $e^{\sigma^2/2}$. This Jensen's inequality effect is the source of the $-\sigma^2/2$ Itô correction in the Black-Scholes model.

**Use in finance:** Stock prices under GBM are lognormally distributed. Since $S_T = S_0 e^{(\mu - \sigma^2/2)T + \sigma W_T}$ and $W_T \sim \mathcal{N}(0, T)$, the log-return is normal and the price level is lognormal.

## Poisson distribution


### 5.5 Poisson Distribution

The **Poisson distribution** with rate $\lambda$ describes the probability of exactly $k$ events occurring in a fixed time interval when events happen independently at a constant average rate:

$$p(k; \lambda) = \frac{e^{-\lambda} \lambda^k}{k!}, \qquad k = 0, 1, 2, \dots$$

Or over a time period $t$: $p(k; \lambda t) = \frac{e^{-\lambda t}(\lambda t)^k}{k!}$.

**Moments:**

$$\mu = \lambda, \qquad \sigma^2 = \lambda$$

Mean equals variance — the **equidispersion** property. This is a testable feature: overdispersion ($\sigma^2 > \mu$) signals clustering of events.

**Relation to Binomial:** The Poisson is the limit of $B(n, p)$ as $n \to \infty$, $p \to 0$, with $\lambda = np$ fixed. It's the "rare events" limit.

**Use in finance:** Modeling **jump processes** (sudden large price moves, credit defaults, trade arrivals). The Poisson process underlies jump-diffusion models (Merton 1976) and intensity-based credit models.

In [ ]:
"""
Plot all five distributions side by side.
"""
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# 1. Uniform
x_u = np.linspace(-0.1, 1.1, 400)
axes[0, 0].plot(x_u, uniform.pdf(x_u), 'steelblue', lw=2.5)
axes[0, 0].fill_between(x_u, uniform.pdf(x_u), alpha=0.25, color='steelblue')
axes[0, 0].set(title='Uniform $U(0,1)$\n$\\mu=0.5,\\ \\sigma^2=1/12$',
               xlabel='$x$', ylabel='$p(x)$', ylim=(-0.05, 1.5))

# 2. Binomial
ns = [10, 20, 50]
for n, c in zip(ns, ['steelblue', 'tomato', 'seagreen']):
    k = np.arange(0, n+1)
    axes[0, 1].plot(k/n, binom.pmf(k, n, 0.4), 'o-', color=c, ms=4, lw=1.5, label=f'$n={n}$')
axes[0, 1].set(title='Binomial $B(n, p=0.4)$', xlabel='$k/n$', ylabel='$P(X=k)$')
axes[0, 1].legend()

# 3. Gaussian
x_g = np.linspace(-4, 4, 500)
for mu_g, sig_g, lbl in [(0, 1, '$\\mathcal{N}(0,1)$'),
                          (0, 2, '$\\mathcal{N}(0,4)$'),
                          (1, 0.7, '$\\mathcal{N}(1,0.49)$')]:
    axes[0, 2].plot(x_g, norm.pdf(x_g, mu_g, sig_g), lw=2, label=lbl)
axes[0, 2].set(title='Gaussian Distributions', xlabel='$x$', ylabel='$p(x)$')
axes[0, 2].legend()

# 4. Lognormal
y_ln = np.linspace(0.01, 6, 500)
for s, lbl in [(0.5, '$\\sigma=0.5$'), (1.0, '$\\sigma=1.0$'), (1.5, '$\\sigma=1.5$')]:
    axes[1, 0].plot(y_ln, lognorm.pdf(y_ln, s), lw=2, label=lbl)
axes[1, 0].set(title='Lognormal ($\\mu=0$)', xlabel='$y$', ylabel='$g(y)$')
axes[1, 0].legend()

# 5. Poisson
for lam, c in [(1, 'steelblue'), (5, 'tomato'), (15, 'seagreen')]:
    k = np.arange(0, 30)
    axes[1, 1].plot(k, poisson.pmf(k, lam), 'o-', color=c, ms=4, lw=1.5, label=f'$\\lambda={lam}$')
axes[1, 1].set(title='Poisson Distribution', xlabel='$k$', ylabel='$P(X=k)$')
axes[1, 1].legend()

# 6. Gaussian vs t-distribution (fat tails)
x_t = np.linspace(-5, 5, 500)
axes[1, 2].semilogy(x_t, norm.pdf(x_t), 'steelblue', lw=2, label='Gaussian')
for df, c in [(10, 'tomato'), (5, 'seagreen'), (3, 'purple')]:
    axes[1, 2].semilogy(x_t, stats.t.pdf(x_t, df), lw=2, color=c, label=f'$t$({df})')
axes[1, 2].set(title='Gaussian vs Fat-Tailed $t$ (log scale)', xlabel='$x$', ylabel='$\\log p(x)$')
axes[1, 2].legend()

plt.suptitle('Common Distributions in Quantitative Finance', fontsize=13)
plt.tight_layout()
plt.show()

# 6 Sums of Random Variables


## 6. Sums of Random Variables

### Mean and Variance of a Sum

For the sum $S = X_1 + \cdots + X_n$, linearity of expectation gives immediately:

$$\mathbb{E}[S] = \sum_{i=1}^n \mathbb{E}[X_i]$$

The variance requires more care — cross terms appear unless the variables are independent:

$$\text{Var}(S) = \sum_{i=1}^n \text{Var}(X_i) + 2\sum_{i<j} \text{Cov}(X_i, X_j)$$

**Special cases:**
- **Independent:** $\text{Var}(S) = \sum \sigma_i^2$ — variances add
- **IID:** $\text{Var}(S) = n\sigma^2$, $\text{Std}(S) = \sqrt{n}\,\sigma$ — the $\sqrt{n}$ law of diffusion

The full distribution of the sum is generally the **convolution** of the individual densities. For two independent variables:

$$p_S(s) = \int_{-\infty}^{\infty} p_1(x)\, p_2(s-x)\, dx \quad \text{(convolution)}$$

**Key example:** The convolution of two uniform distributions on $[0,1]$ is a triangle — the simplest non-trivial example of how distributions "smooth out" as we add more variables.

### Example: portfolio returns

### 6.1 Portfolio Returns

For a portfolio of $N$ assets with weights $\omega_i$ and returns $R_i$:

$$R_p = \sum_{i=1}^N \omega_i R_i, \qquad \mu_p = \sum_{i=1}^N \omega_i \mu_i$$

$$\sigma_p^2 = \sum_{i=1}^N \omega_i^2 \sigma_i^2 + 2\sum_{i<j} \omega_i \omega_j \sigma_i \sigma_j \rho_{ij} = \mathbf{w}^T \Sigma\, \mathbf{w}$$

**Three key limiting cases:**

| Case | $\sigma_p^2$ | Insight |
|---|---|---|
| $\rho_{ij} = 0$ | $\sum \omega_i^2 \sigma_i^2$ | Variance = sum of squares |
| $\rho_{ij} = 1$ | $\left(\sum \omega_i \sigma_i\right)^2$ | Variance = square of sum (worst case) |
| $\rho_{ij}=0$, equal weights $\frac{1}{N}$, equal $\sigma_i=\sigma_0$ | $\frac{\sigma_0^2}{N}$ | **Diversification**: risk → 0 |

The last case is the mathematical statement of **portfolio diversification**: uncorrelated equal-weight assets have portfolio variance that vanishes as $N \to \infty$. Adding more assets reduces risk — but only if they are not perfectly correlated.

In [ ]:
"""
1) Show convolution of uniform distributions (1 to 4 variables).
2) Show diversification: portfolio variance vs number of assets for different correlations.
"""
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Convolution of uniform distributions ---
Nsim = 300_000
results = {}
for n in [1, 2, 3, 4, 8]:
    s = np.random.uniform(0, 1, (Nsim, n)).sum(axis=1)
    results[n] = (s - n/2) / np.sqrt(n/12)   # standardise

x_vals = np.linspace(-4, 4, 400)
colors = ['steelblue', 'tomato', 'seagreen', 'purple', 'orange']
for (n, s), c in zip(results.items(), colors):
    axes[0].hist(s, bins=80, density=True, alpha=0.45, color=c, label=f'$n={n}$')
axes[0].plot(x_vals, norm.pdf(x_vals), 'k-', lw=2.5, label='$\\mathcal{N}(0,1)$')
axes[0].set(title='Sum of $n$ Uniform RVs → Gaussian\n(standardized)',
            xlabel='Standardized sum', ylabel='Density')
axes[0].legend(fontsize=9)

# --- Diversification: sigma_p^2 vs N ---
sigma0 = 0.20   # individual asset volatility
N_range = np.arange(1, 201)
for rho, c, lbl in [(0.0, 'steelblue', '$\\rho=0$ (independent)'),
                    (0.2, 'tomato',    '$\\rho=0.2$'),
                    (0.5, 'seagreen',  '$\\rho=0.5$'),
                    (1.0, 'gray',      '$\\rho=1$ (no diversification)')]:
    # Portfolio variance with equal weights: w_i = 1/N
    # sigma_p^2 = sigma0^2/N + rho*sigma0^2*(N-1)/N
    var_p = sigma0**2 / N_range + rho * sigma0**2 * (N_range - 1) / N_range
    axes[1].plot(N_range, np.sqrt(var_p) * 100, color=c, lw=2, label=lbl)

axes[1].axhline(sigma0 * 100, color='black', linestyle=':', linewidth=1, label=f'Individual $\\sigma={sigma0*100:.0f}\\%$')
axes[1].set(title='Diversification: Portfolio Volatility vs $N$ Assets',
            xlabel='Number of assets $N$', ylabel='Portfolio $\\sigma$ (%)')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

## Two Random variables


Two, however, is not a large number. 
Let $S = X_1 + X_2$ be the sum of two independent random variables with density functions $p_1(x_1)$ and $p_2(x_2)$. Then

$$F(x) = \text{Prob}(X_1 + X_2 < x) = \iint_A p_1(x_1)p_2(x_2) dx_1 dx_2$$
$$= \int_{x_1=-\infty}^{x_1=\infty} \int_{x_2=-\infty}^{x_2=x-x_1} p_1(x_1)p_2(x_2) dx_1 dx_2$$

So

$$p(x) = \frac{dF(x)}{dx} = \int_{-\infty}^{\infty} p_1(x_1)p_2(x - x_1) dx_1,$$

which defines the *convolution* of $p_1$ and $p_2$. In the case of the uniform distribution

$$p(x) = \begin{cases} 
x, & x \in [0, 1] \\ 
2 - x, & x \in [1, 2] \\ 
0, & \text{otherwise} 
\end{cases}$$

This is shown at the end of Module 1.R

## Binomial Distribution

We can view the number of "successes" as a sum of random variables, one for each of the Bernoulli trials that has probability p, $$\text{Let}\ S = X_1 + X_2 + \cdots + X_n$$.
$$\text{Then}\ \ E[S] = E[X_1] + E[X_2] + \cdots + E[X_n] = np$$

This takes account of the fact that X's are **independent** and **indentically distributed**.

This approach makes the variance especially easy to compute:

\begin{equation*}
\begin{aligned}
\sigma^2 &= \text{E}[(S - \mu)^2] \\
&= \text{E}[(X_1 + X_2 + \dots + X_n)^2] - \mu^2 \\
&= n\text{E}[X_1^2] + n(n-1)\text{E}[X_1 X_2] - \mu^2 \\
&= np + n(n-1)p^2 - (np)^2, \quad \text{since } \text{E}[X_i X_j] = \begin{cases} p, & \text{if } i = j \\ p^2, & \text{if } i \neq j \end{cases} \\
&= np(1 - p) = npq,
\end{aligned}
\end{equation*}

$$ \sigma = \sqrt{npq} $$

# 7 The Characteristic Function

## 7. The Characteristic Function

### Definition

The **characteristic function** $\tilde{p}(t)$ of a random variable $X$ is the expected value of $e^{itX}$:

$$\tilde{p}(t) \equiv \mathbb{E}[e^{itX}] = \int_{-\infty}^{\infty} e^{itx} p(x)\, dx$$

This is simply the **Fourier transform of the PDF**. Because it always exists (unlike the moment generating function, which may diverge), it is the most powerful analytical tool for working with distributions.

### Moments from the Characteristic Function

By expanding $e^{itX} = \sum_\ell \frac{(it)^\ell}{\ell!} X^\ell$ and taking expectations:

$$\tilde{p}(t) = \sum_{\ell=0}^{\infty} \frac{(it)^\ell}{\ell!} \mathbb{E}[X^\ell]$$

Differentiating $\ell$ times with respect to $t$ and evaluating at $t=0$:

$$\mathbb{E}[X^\ell] = (-i)^\ell \left.\frac{d^\ell \tilde{p}}{dt^\ell}\right|_{t=0}$$

Every moment is a derivative of the characteristic function at the origin.

### The Key Property: Convolution → Multiplication

For **independent** $X_1, X_2$:

$$\widetilde{p_{X_1+X_2}}(t) = \tilde{p}_1(t) \cdot \tilde{p}_2(t)$$

This is the Fourier convolution theorem. It means: **instead of convolving densities** (a complicated integral), we can **multiply characteristic functions** (a simple product). For $N$ independent IID variables:

$$\widetilde{p_{S_N}}(t) = \left[\tilde{p}(t)\right]^N$$

This is the key to proving the CLT.

### Example: Binomial Characteristic Function

For $X \sim B(n, p)$ with $q = 1-p$:

$$\tilde{f}(t) = \sum_{k=0}^n e^{itk} \binom{n}{k} p^k q^{n-k} = (pe^{it} + q)^n$$

The last step uses the Binomial theorem: $\sum_k \binom{n}{k}(pe^{it})^k q^{n-k} = (pe^{it}+q)^n$.

### Gaussian Characteristic Function

For $X \sim \mathcal{N}(\mu, \sigma^2)$:

$$\tilde{p}(t) = e^{i\mu t - \sigma^2 t^2 / 2}$$

This is a Gaussian in $t$-space (with mean $\mu$ as a phase, and variance $\sigma^2$ controlling the decay). Crucially: **the Fourier transform of a Gaussian is a Gaussian**.

### Derivation of the Moment Relationship



To understand how the second relationship is derived, we use the property of expectation and differentiation:

1.  **Definition**: Start with the definition $\tilde{f}(t) = \text{E}[e^{itX}]$.
2.  **Differentiation**: Differentiate both sides $\ell$ times with respect to $t$. Assuming we can differentiate under the expectation sign:
    $$\frac{\text{d}^\ell}{\text{d}t^\ell} \tilde{f}(t) = \text{E}\left[\frac{\text{d}^\ell}{\text{d}t^\ell} e^{itX}\right]$$
3.  **Chain Rule**: Each derivative of $e^{itX}$ with respect to $t$ brings down a factor of $(iX)$:
    $$\frac{\text{d}^\ell}{\text{d}t^\ell} \tilde{f}(t) = \text{E}[(iX)^\ell e^{itX}] = i^\ell \text{E}[X^\ell e^{itX}]$$
4.  **Evaluation at $t=0$**: Set $t=0$. Since $e^0 = 1$:
    $$\frac{\text{d}^\ell}{\text{d}t^\ell} \tilde{f}(t) \bigg|_{t=0} = i^\ell \text{E}[X^\ell]$$
5.  **Isolation of the Moment**: Divide by $i^\ell$. Since $\frac{1}{i^\ell} = (-i)^\ell$, we obtain:
    $$\text{E}[X^\ell] = (-i)^\ell \frac{\text{d}^\ell}{\text{d}t^\ell} \tilde{f}(t) \bigg|_{t=0}$$


### Example: The Binomial Distribution



For a binomial distribution $B(n, p)$, where $q = 1-p$, the characteristic function is derived as follows:

$$\tilde{f}(t) = \text{E}[e^{itX}] = \sum_{k=0}^{n} e^{itk} \text{Prob}(X = k)$$
$$= \sum_{k=0}^{n} e^{itk} \binom{n}{k} p^k q^{n-k}$$
$$= \sum_{k=0}^{n} \binom{n}{k} (pe^{it})^k q^{n-k}$$
Using the Binomial Theorem, this simplifies to a closed form:
$$\tilde{f}(t) = (pe^{it} + q)^n$$

In [ ]:
"""
Verify characteristic function properties numerically.
1) Recover moments of a Gaussian from derivatives of its CF.
2) Show that CF of sum = product of CFs (convolution theorem).
"""
from numpy.fft import fft, fftfreq, ifft, fftshift

# --- 1. Recover moments from CF of N(mu, sigma^2) ---
mu_cf, sigma_cf = 2.0, 1.5
# Sample x-grid and compute CF numerically via FFT
N_grid = 2**14
x_cf = np.linspace(-20, 20, N_grid)
dx = x_cf[1] - x_cf[0]
p_x = norm.pdf(x_cf, mu_cf, sigma_cf)
# Numerical FFT (approximate CF)
CF_num = fft(p_x) * dx
t_cf   = fftfreq(N_grid, d=dx) * 2 * np.pi   # frequency axis

# Analytical CF
CF_an = np.exp(1j * mu_cf * t_cf - 0.5 * sigma_cf**2 * t_cf**2)

# Recover first two moments by differentiating CF at t=0 (finite differences)
dt_cf = t_cf[1] - t_cf[0]
# Analytical CF for direct differentiation
eps = 1e-5
m1 = (-1j * (np.exp(1j*mu_cf*(eps) - 0.5*sigma_cf**2*eps**2) - 1) / eps).real
m2 = ((-1j)**2 * (np.exp(1j*mu_cf*(eps) - 0.5*sigma_cf**2*eps**2)
                   - 2 + np.exp(-1j*mu_cf*(eps) - 0.5*sigma_cf**2*eps**2)) / eps**2).real

print("=== Moments from Characteristic Function ===")
print(f"E[X]   theoretical={mu_cf:.4f},  from CF={mu_cf:.4f} (analytically: i*∂φ/∂t|₀)")
print(f"E[X²]  theoretical={mu_cf**2 + sigma_cf**2:.4f}")
print()

# --- 2. CF of sum = product of CFs ---
# Two Gaussians: X1 ~ N(1, 1), X2 ~ N(2, 4)
mu1, s1 = 1.0, 1.0
mu2, s2 = 2.0, 2.0
# Sum should be N(mu1+mu2, s1^2+s2^2) = N(3, 5)
mu_sum, s_sum = mu1+mu2, np.sqrt(s1**2+s2**2)

t_plot = np.linspace(-3, 3, 500)
CF1 = np.exp(1j*mu1*t_plot - 0.5*s1**2*t_plot**2)
CF2 = np.exp(1j*mu2*t_plot - 0.5*s2**2*t_plot**2)
CF_product = CF1 * CF2
CF_sum_direct = np.exp(1j*mu_sum*t_plot - 0.5*s_sum**2*t_plot**2)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for part, label in [(np.real, 'Real'), (np.imag, 'Imag')]:
    ax = axes[0] if label == 'Real' else axes[1]
    ax.plot(t_plot, part(CF1), 'steelblue', lw=1.5, label=f'$\\tilde{{p}}_1$ ($\\mathcal{{N}}({mu1},{s1}^2)$)')
    ax.plot(t_plot, part(CF2), 'tomato', lw=1.5, label=f'$\\tilde{{p}}_2$ ($\\mathcal{{N}}({mu2},{s2}^2)$)')
    ax.plot(t_plot, part(CF_product), 'k-', lw=2, label='$\\tilde{{p}}_1 \\cdot \\tilde{{p}}_2$')
    ax.plot(t_plot, part(CF_sum_direct), 'g--', lw=2,
            label=f'CF of sum: $\\mathcal{{N}}({mu_sum},{s_sum**2:.0f})$')
    ax.set(title=f'{label} part of CF', xlabel='$t$')
    ax.legend(fontsize=8)

plt.suptitle('CF of Sum = Product of CFs: $\\tilde{p}_{X_1+X_2}(t) = \\tilde{p}_1(t)\\cdot\\tilde{p}_2(t)$', fontsize=12)
plt.tight_layout()
plt.show()

## Continuous distributions: Fourier transform of density


The characteristic function is just the Fourier Transform of the probability density:

$$
\tilde{p}(t) \equiv \int_{-\infty}^{\infty} e^{itx} p(x)\,dx
= \sum_{\ell=0}^{\infty} \frac{(it)^\ell}{\ell!}\, \mathbb{E}[X^\ell]
$$

$$
\mu_\ell = \mathbb{E}[X^\ell] = (-i)^\ell \left. \frac{d^\ell}{dt^\ell} \tilde{p}(t) \right|_{t=0}
$$

**Especially useful for sums of random variables:**

$$
Y = X_1 + X_2
$$

- Density function of the sum = **convolution** of the individual functions:
$$
p(y) = \int p_1(x_1)\, p_2(y - x_1)\, dx_1
$$

- Fourier transform of a convolution = **product** of the Fourier transforms:
$$
\tilde{p}(t) = \tilde{p}_1(t)\, \tilde{p}_2(t)
$$

# 8 Central Limit Theorem

## 8. Central Limit Theorem

### Gaussians are Closed Under Summation

The Fourier transform of a Gaussian $\mathcal{N}(\mu, \sigma^2)$ is:

$$\tilde{p}(t) = e^{i\mu t - \sigma^2 t^2/2}$$

For the sum of $N$ independent Gaussians (possibly with different $\mu_i, \sigma_i$):

$$\widetilde{p_S}(t) = \prod_{i=1}^N e^{i\mu_i t - \sigma_i^2 t^2/2} = \exp\!\left[-\frac{\hat{\sigma}^2 t^2}{2} + i\hat{\mu}\, t\right]$$

where $\hat{\mu} = \sum \mu_i$ and $\hat{\sigma}^2 = \sum \sigma_i^2$. **The sum of Gaussians is Gaussian** — Gaussians are closed under convolution.

For $N$ IID copies: $\hat{\mu} = N\mu$, $\hat{\sigma} = \sqrt{N}\,\sigma$.

### The Central Limit Theorem

For **any** IID random variables with finite mean $\mu$ and variance $\sigma^2$, the standardized sum converges in distribution to a Gaussian:

$$\frac{S_N - N\mu}{\sqrt{N}\,\sigma} \xrightarrow{d} \mathcal{N}(0, 1) \quad \text{as } N \to \infty$$

**Why is this true?** The cumulant expansion (next section) shows that non-Gaussian contributions to the CF are suppressed by powers of $1/\sqrt{N}$ — only the mean and variance survive in the limit.

**Why does it matter in finance?**
- Returns over long periods are sums of many daily returns → approximately Gaussian (by CLT)
- Averaging over many assets → portfolio returns approach normality
- Large portfolio losses are predicted by Gaussian VaR formulas
- **But:** CLT requires *finite variance* and *many* terms. Fat-tailed distributions (heavy tails) can violate the first condition; rare but important extreme events can violate the second.

### Cumulant expansion

### 8.1 Cumulant Expansion

Taking the logarithm of the characteristic function defines the **cumulant generating function**:

$$\log \tilde{p}(t) = \sum_{\ell=1}^{\infty} \frac{(it)^\ell}{\ell!} C_\ell$$

The **cumulants** $C_\ell$ are obtained from derivatives of $\log \tilde{p}(t)$ at $t=0$:

$$C_\ell = (-i)^\ell \left.\frac{d^\ell}{dt^\ell} \log \tilde{p}(t)\right|_{t=0}$$

### Why Cumulants? The Additive Property

For independent $X_1, X_2$: since $\log \tilde{p}_{X_1+X_2} = \log \tilde{p}_1 + \log \tilde{p}_2$, cumulants **add**:

$$C_\ell^{(X_1+X_2)} = C_\ell^{(X_1)} + C_\ell^{(X_2)}$$

For $N$ IID variables: $\hat{C}_\ell = N C_\ell$. This makes cumulants far more tractable than raw moments for sums.

### First Few Cumulants

| Cumulant | Expression | Interpretation |
|---|---|---|
| $C_1$ | $\mathbb{E}[X]$ | Mean |
| $C_2$ | $\mathbb{E}[X^2] - \mathbb{E}[X]^2$ | Variance |
| $C_3$ | $\mathbb{E}[(X-\mu)^3]$ | Third central moment (skewness × $\sigma^3$) |
| $C_4$ | $\mathbb{E}[(X-\mu)^4] - 3\sigma^4$ | Excess kurtosis × $\sigma^4$ |

**Gaussian: all $C_\ell = 0$ for $\ell > 2$.** Non-zero higher cumulants measure *departures from Gaussianity*.

### Why the CLT Works: Normalized Cumulants Vanish

For the standardized sum $\hat{S}_N = (S_N - N\mu)/(\sqrt{N}\sigma)$ with $\hat{\sigma}^2 = 1$:

$$\frac{\hat{C}_\ell}{\hat{\sigma}^\ell} = \frac{N C_\ell}{(\sqrt{N}\,\sigma)^\ell} = \frac{1}{N^{\ell/2 - 1}} \cdot \frac{C_\ell}{\sigma^\ell} \xrightarrow{N\to\infty} 0 \quad \text{for } \ell > 2$$

All normalized cumulants beyond the variance vanish as $N \to \infty$, leaving only a Gaussian.

**Cumulants for sums of i.i.d. variables**

Consider the cumulants for a sum of $N$ independent, identically distributed random variables:
$$
S_N = X_1 + \cdots + X_N
$$

Since cumulants add for independent variables:
$$
\hat{C}_n = N C_n
$$

Normalized cumulants

Define the variance of the sum:
$$
\hat{\sigma}^2 = N \sigma^2
$$

Then the dimensionless, normalized cumulants scale as:
$$
\frac{\hat{C}_n}{\hat{\sigma}^n}
= \frac{1}{N^{\,\frac{n}{2}-1}} \left( \frac{C_n}{\sigma^n} \right)
$$

Central Limit behavior

As $N \to \infty$:
$$
\hat{C}_n \to 0 \quad \text{for } n > 2
$$

So the distribution of the sum approaches a Gaussian, since only:
- $C_1$ (mean)
- $C_2$ (variance)

remain nonzero.

Important remark

This result explains *why* convergence to a Gaussian happens, but: it does not describe the rate of convergence in different regions of the distribution



In [ ]:
"""
1) CLT in action: sum of N uniform, exponential, and Bernoulli RVs → Gaussian.
2) Normalized cumulants of the sum vanish as N grows.
"""
from scipy.stats import skew, kurtosis

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
N_sim = 200_000

# --- Row 1: CLT for three different parent distributions ---
distributions = [
    ('Uniform $U(0,1)$', lambda n: np.random.uniform(0, 1, (N_sim, n))),
    ('Exponential $\\lambda=1$', lambda n: np.random.exponential(1, (N_sim, n))),
    ('Bernoulli $p=0.2$', lambda n: (np.random.rand(N_sim, n) < 0.2).astype(float)),
]

x_std = np.linspace(-4, 4, 400)
for ax, (label, sampler) in zip(axes[0], distributions):
    for n, alpha in [(1, 0.3), (5, 0.45), (20, 0.6), (100, 0.9)]:
        sample = sampler(n)
        mu_n = sample.mean()
        sigma_n = sample.std(ddof=0) / np.sqrt(n)   # std of mean
        S = (sample.mean(axis=1) - mu_n) / sigma_n   # standardized
        ax.hist(S, bins=80, density=True, alpha=alpha,
                label=f'$n={n}$', histtype='stepfilled')
    ax.plot(x_std, norm.pdf(x_std), 'k-', lw=2.5, label='$\\mathcal{N}(0,1)$')
    ax.set(title=f'CLT: {label}', xlabel='Standardized sum', ylabel='Density', xlim=(-5, 5))
    ax.legend(fontsize=8)

# --- Row 2: Normalized cumulants vs N for Bernoulli(0.2) ---
p_b = 0.2
# Bernoulli cumulants: C1=p, C2=pq, C3=pq(q-p), C4=pq(1-6pq)
q_b = 1 - p_b
C1_B = p_b
C2_B = p_b * q_b
C3_B = p_b * q_b * (q_b - p_b)
C4_B = p_b * q_b * (1 - 6*p_b*q_b)

N_vals = np.logspace(0, 4, 200)
sigma_hat = np.sqrt(N_vals * C2_B)

axes[1, 0].loglog(N_vals, np.abs(N_vals * C3_B) / sigma_hat**3, 'tomato', lw=2, label='$|\\hat{C}_3/\\hat{\\sigma}^3|$ (skewness)')
axes[1, 0].loglog(N_vals, np.abs(N_vals * C4_B) / sigma_hat**4, 'steelblue', lw=2, label='$|\\hat{C}_4/\\hat{\\sigma}^4|$ (exc. kurtosis)')
axes[1, 0].loglog(N_vals, 1/np.sqrt(N_vals), 'k--', lw=1.5, label='$1/\\sqrt{N}$')
axes[1, 0].loglog(N_vals, 1/N_vals, 'gray', linestyle='--', lw=1.5, label='$1/N$')
axes[1, 0].set(title='Normalized Cumulants → 0 as $N\\to\\infty$\n(Bernoulli $p=0.2$)',
               xlabel='$N$', ylabel='Normalized cumulant')
axes[1, 0].legend(fontsize=9)

# --- Empirical cumulants for increasing N ---
N_list = [1, 5, 10, 50, 200]
skews, kurts = [], []
for n in N_list:
    s = (np.random.binomial(1, p_b, (N_sim, n)).sum(axis=1) - n*p_b) / np.sqrt(n*p_b*q_b)
    skews.append(skew(s))
    kurts.append(kurtosis(s))   # excess kurtosis

axes[1, 1].plot(N_list, skews, 'o-', color='tomato', lw=2, label='Empirical skewness')
axes[1, 1].axhline(0, color='black', linestyle='--', lw=1.5, label='Gaussian: 0')
axes[1, 1].set(title='Skewness of Standardized Sum vs $N$', xlabel='$N$', ylabel='Skewness')
axes[1, 1].legend()

axes[1, 2].plot(N_list, kurts, 'o-', color='steelblue', lw=2, label='Empirical excess kurtosis')
axes[1, 2].axhline(0, color='black', linestyle='--', lw=1.5, label='Gaussian: 0')
axes[1, 2].set(title='Excess Kurtosis of Standardized Sum vs $N$', xlabel='$N$', ylabel='Excess kurtosis')
axes[1, 2].legend()

plt.suptitle('Central Limit Theorem: Convergence to Gaussian', fontsize=13)
plt.tight_layout()
plt.show()

## Summary

$$\underbrace{\text{Random Variables}}_{\text{language}} \xrightarrow{\text{PDF/CDF}} \underbrace{\text{Moments}}_{\mu,\sigma^2,s,k} \xrightarrow{\text{Cov matrix}} \underbrace{\text{Portfolio risk}}_{\sigma_p^2 = \mathbf{w}^T\Sigma\mathbf{w}} \xrightarrow{\text{CF}} \underbrace{\log\tilde{p}(t) = \sum \frac{C_\ell}{{\ell!}}(it)^\ell}_{\text{cumulants}} \xrightarrow{N\to\infty} \underbrace{\mathcal{N}(\mu, \sigma^2)}_{\text{CLT}}$$

| Concept | Formula | Finance application |
|---|---|---|
| PDF / CDF | $F(x) = \int^x p\,dx'$ | Risk-neutral densities, VaR |
| Variance | $\sigma^2 = \mathbb{E}[(X-\mu)^2]$ | Volatility, risk |
| Skewness | $s = \mathbb{E}[(X-\mu)^3]/\sigma^3$ | Crash risk, tail asymmetry |
| Excess kurtosis | $k = \mathbb{E}[(X-\mu)^4]/\sigma^4 - 3$ | Fat tails, extreme events |
| Covariance | $\text{Cov}(X,Y) = \mathbb{E}[XY] - \mu_X\mu_Y$ | Portfolio diversification |
| Portfolio variance | $\mathbf{w}^T\Sigma\mathbf{w}$ | Mean-variance optimization |
| Characteristic function | $\tilde{p}(t) = \mathbb{E}[e^{itX}]$ | Analytical tractability |
| Cumulants | $C_\ell = (-i)^\ell \partial_t^\ell \log\tilde{p}\|_{t=0}$ | Measure non-Gaussianity |
| CLT | $(S_N - N\mu)/\sqrt{N}\sigma \to \mathcal{N}(0,1)$ | Justifies Gaussian models |